In [0]:
%sql
-- Databricks notebook source
-- MAGIC %md
-- MAGIC # 03 · Gold: reliability metrics
-- MAGIC
-- MAGIC Business-ready tables built from Silver. These answer the questions an SRE / engineering manager asks:
-- MAGIC
-- MAGIC | Gold table | Question it answers |
-- MAGIC |---|---|
-- MAGIC | `service_health_hourly` | How healthy was each service each hour? Was there an error spike? |
-- MAGIC | `incident_metrics` | For each incident: how fast did we detect, acknowledge and resolve it? What broke, who was hit? |
-- MAGIC | `deployment_impact` | Did this deployment make things worse? Did it cause an incident? |
-- MAGIC | `service_reliability` | Per service: change failure rate, MTTA, MTTR, SLA breaches (DORA-style scorecard) |
-- MAGIC
-- MAGIC **Refresh strategy: full rebuild (`CREATE OR REPLACE`).** Gold is small aggregates over Silver, and
-- MAGIC late-arriving events can change *past* hours. Recomputing everything is simple and always correct.
-- MAGIC At larger scale you'd switch to recomputing only the affected hours with `MERGE` (see the note at the end).

-- COMMAND ----------

CREATE SCHEMA IF NOT EXISTS incident_analytics.gold;
USE CATALOG incident_analytics;

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ## 1 · Service health per hour
-- MAGIC **Spike rule:** error rate ≥ 5% in an hour with at least 50 requests.
-- MAGIC Normal hours run at about 1% errors. The volume floor stops a sparse hour (e.g. 3 errors out of 12 late events) from
-- MAGIC looking like a spike. `baseline_error_rate` (median of the previous 24 normal-volume hours) is kept as context.

-- COMMAND ----------

CREATE OR REPLACE TABLE gold.service_health_hourly
COMMENT 'One row per service per hour: volume, errors, latency, spike flag'
AS
WITH hourly AS (
  SELECT
    event_hour,
    service,
    COUNT(*)                                  AS total_requests,
    COUNT_IF(level = 'ERROR')                 AS error_count,
    COUNT_IF(level = 'WARN')                  AS warn_count,
    COUNT_IF(status_code >= 500)              AS http_5xx_count,
    COUNT(DISTINCT user_id)                   AS distinct_users,
    PERCENTILE_APPROX(latency_ms, 0.50)       AS p50_latency_ms,
    PERCENTILE_APPROX(latency_ms, 0.95)       AS p95_latency_ms,
    MODE(error_code)                          AS top_error_code       -- most frequent; NULLs ignored
  FROM silver.app_logs
  GROUP BY event_hour, service
),
rated AS (
  SELECT *,
         ROUND(error_count / total_requests, 4) AS error_rate,
         total_requests >= 50                   AS is_normal_volume
  FROM hourly
)
SELECT
  r.*,
  ROUND(PERCENTILE_APPROX(CASE WHEN is_normal_volume THEN error_rate END, 0.5) OVER (
          PARTITION BY service ORDER BY event_hour
          ROWS BETWEEN 24 PRECEDING AND 1 PRECEDING), 4)           AS baseline_error_rate,
  is_normal_volume AND error_rate >= 0.05                         AS is_error_spike
FROM rated r;

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ## 2 · Incident metrics
-- MAGIC - **Time to detect:** bad deployment → incident opened
-- MAGIC - **Time to acknowledge (MTTA):** opened → first `INVESTIGATING` (from the status history table)
-- MAGIC - **Time to resolve (MTTR):** opened → resolved
-- MAGIC - **Blast radius:** error events and distinct users hit between the deployment and the resolution
-- MAGIC - **SLA:** resolution targets by severity: SEV1 60 min, SEV2 240 min, SEV3 1,440 min

-- COMMAND ----------

CREATE OR REPLACE TABLE gold.incident_metrics
COMMENT 'One row per incident: detect / acknowledge / resolve times, blast radius, root error, SLA'
AS
WITH ack AS (
  SELECT incident_id, MIN(updated_at) AS acknowledged_at
  FROM silver.incident_status_history
  WHERE status = 'INVESTIGATING'
  GROUP BY incident_id
),
impact AS (
  SELECT
    i.incident_id,
    COUNT(*)                  AS error_events,
    COUNT(DISTINCT l.user_id) AS affected_users,
    MODE(l.error_code)        AS dominant_error_code
  FROM silver.incidents i
  JOIN silver.deployments d
    ON d.deployment_id = i.related_deployment_id
  JOIN silver.app_logs l
    ON  l.service = i.service
    AND l.level = 'ERROR'
    AND l.event_ts BETWEEN d.deployed_at AND COALESCE(i.resolved_at, current_timestamp())
  GROUP BY i.incident_id
),
rollback AS (
  SELECT i.incident_id,
         MIN_BY(rb.deployment_id, rb.deployed_at) AS rollback_deployment_id,
         MIN(rb.deployed_at)                      AS rolled_back_at
  FROM silver.incidents i
  JOIN silver.deployments rb
    ON  rb.service = i.service
    AND rb.change_type = 'rollback'
    AND rb.deployed_at BETWEEN i.opened_at AND COALESCE(i.resolved_at, current_timestamp())
  GROUP BY i.incident_id
),
base AS (
  SELECT
    i.incident_id, i.service, i.severity, i.status, i.assigned_team, i.title,
    i.related_deployment_id, d.version AS bad_version, d.deployed_at,
    i.opened_at, a.acknowledged_at, rb.rolled_back_at, i.resolved_at,
    rb.rollback_deployment_id,
    ROUND(timestampdiff(SECOND, d.deployed_at, i.opened_at)       / 60.0, 1) AS minutes_to_detect,
    ROUND(timestampdiff(SECOND, i.opened_at,  a.acknowledged_at)  / 60.0, 1) AS minutes_to_acknowledge,
    ROUND(timestampdiff(SECOND, i.opened_at,  i.resolved_at)      / 60.0, 1) AS minutes_to_resolve,
    CASE i.severity WHEN 'SEV1' THEN 60 WHEN 'SEV2' THEN 240 WHEN 'SEV3' THEN 1440 END AS sla_minutes,
    COALESCE(im.error_events, 0)   AS error_events,
    COALESCE(im.affected_users, 0) AS affected_users,
    im.dominant_error_code
  FROM silver.incidents i
  LEFT JOIN silver.deployments d ON d.deployment_id = i.related_deployment_id
  LEFT JOIN ack a                ON a.incident_id   = i.incident_id
  LEFT JOIN impact im            ON im.incident_id  = i.incident_id
  LEFT JOIN rollback rb          ON rb.incident_id  = i.incident_id
)
SELECT *,
       i.status <> 'RESOLVED' AS is_open,
       -- open incidents count as breached once they're past the target
       COALESCE(minutes_to_resolve,
                timestampdiff(SECOND, opened_at, current_timestamp()) / 60.0) > sla_minutes AS sla_breached
FROM base i;

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ## 3 · Deployment impact
-- MAGIC Compares each deployment's hour with the hour before. Rollbacks should show the error rate *falling*.

-- COMMAND ----------

CREATE OR REPLACE TABLE gold.deployment_impact
COMMENT 'One row per deployment: error rate / latency before vs after, spike and incident flags'
AS
SELECT
  d.deployment_id, d.service, d.version, d.change_type, d.deployed_by, d.deployed_at,
  b.error_rate                              AS error_rate_hour_before,
  a.error_rate                              AS error_rate_deploy_hour,
  ROUND(a.error_rate - b.error_rate, 4)     AS error_rate_change,
  b.p95_latency_ms                          AS p95_latency_before,
  a.p95_latency_ms                          AS p95_latency_after,
  COALESCE(a.is_error_spike, FALSE)         AS spike_in_deploy_hour,
  i.incident_id,
  i.severity                                AS incident_severity,
  i.incident_id IS NOT NULL                 AS caused_incident
FROM silver.deployments d
LEFT JOIN gold.service_health_hourly b
  ON b.service = d.service AND b.event_hour = date_trunc('HOUR', d.deployed_at) - INTERVAL 1 HOUR
LEFT JOIN gold.service_health_hourly a
  ON a.service = d.service AND a.event_hour = date_trunc('HOUR', d.deployed_at)
LEFT JOIN silver.incidents i
  ON i.related_deployment_id = d.deployment_id;

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ## 4 · Service reliability scorecard (DORA-style)
-- MAGIC **Change failure rate** = deployments that caused an incident ÷ deployments (rollbacks excluded).
-- MAGIC It's one of the four DORA metrics, alongside deployment frequency, lead time and time to restore.

-- COMMAND ----------

CREATE OR REPLACE TABLE gold.service_reliability
COMMENT 'One row per service: traffic, error rate, change failure rate, MTTA, MTTR, SLA breaches'
AS
WITH health AS (
  SELECT service,
         SUM(total_requests)                                  AS total_requests,
         SUM(error_count)                                     AS total_errors,
         ROUND(1 - SUM(error_count) / SUM(total_requests), 5) AS success_rate,
         COUNT_IF(is_error_spike)                             AS spike_hours,
         MIN(event_hour)                                      AS first_hour,
         MAX(event_hour)                                      AS last_hour
  FROM gold.service_health_hourly
  GROUP BY service
),
deploys AS (
  SELECT service,
         COUNT_IF(change_type <> 'rollback')                    AS deployments,
         COUNT_IF(change_type <> 'rollback' AND caused_incident) AS failed_deployments,
         COUNT_IF(change_type = 'rollback')                     AS rollbacks
  FROM gold.deployment_impact
  GROUP BY service
),
incidents AS (
  SELECT service,
         COUNT(*)                              AS incidents,
         COUNT_IF(is_open)                     AS open_incidents,
         ROUND(AVG(minutes_to_detect), 1)      AS avg_minutes_to_detect,
         ROUND(AVG(minutes_to_acknowledge), 1) AS mtta_minutes,
         ROUND(AVG(minutes_to_resolve), 1)     AS mttr_minutes,
         COUNT_IF(sla_breached)                AS sla_breaches
  FROM gold.incident_metrics
  GROUP BY service
)
SELECT
  h.service,
  h.first_hour, h.last_hour,
  h.total_requests, h.total_errors, h.success_rate, h.spike_hours,
  COALESCE(d.deployments, 0)        AS deployments,
  COALESCE(d.failed_deployments, 0) AS failed_deployments,
  COALESCE(d.rollbacks, 0)          AS rollbacks,
  ROUND(d.failed_deployments / NULLIF(d.deployments, 0), 3) AS change_failure_rate,
  COALESCE(i.incidents, 0)          AS incidents,
  COALESCE(i.open_incidents, 0)     AS open_incidents,
  i.avg_minutes_to_detect, i.mtta_minutes, i.mttr_minutes,
  COALESCE(i.sla_breaches, 0)       AS sla_breaches
FROM health h
LEFT JOIN deploys d   USING (service)
LEFT JOIN incidents i USING (service)
ORDER BY change_failure_rate DESC NULLS LAST;

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ## Checks

-- COMMAND ----------

-- Detection check: every incident should have a spike in its deployment hour,
-- and every spike hour should belong to an incident. Both columns should be 0.
SELECT
  (SELECT COUNT(*) FROM gold.deployment_impact
     WHERE caused_incident AND NOT spike_in_deploy_hour)                AS incidents_without_spike,
  (SELECT COUNT(*) FROM gold.service_health_hourly h
     WHERE is_error_spike AND NOT EXISTS (
       SELECT 1 FROM gold.incident_metrics i
       WHERE i.service = h.service
         AND h.event_hour >= date_trunc('HOUR', i.deployed_at)
         AND h.event_hour <  COALESCE(i.resolved_at, current_timestamp())))  AS spikes_without_incident;

-- COMMAND ----------

SELECT * FROM gold.service_reliability;

-- COMMAND ----------

SELECT incident_id, service, severity, bad_version, dominant_error_code,
       minutes_to_detect, minutes_to_acknowledge, minutes_to_resolve, sla_minutes, sla_breached,
       error_events, affected_users, rollback_deployment_id
FROM gold.incident_metrics
ORDER BY opened_at;

-- COMMAND ----------

-- Deployments that hurt: biggest error-rate jumps
SELECT deployment_id, service, version, change_type, error_rate_hour_before, error_rate_deploy_hour,
       error_rate_change, caused_incident, incident_id
FROM gold.deployment_impact
ORDER BY error_rate_change DESC NULLS LAST
LIMIT 10;

-- COMMAND ----------

-- Team view: who's on the hook, and how fast do they recover?
SELECT assigned_team, severity,
       COUNT(*) AS incidents,
       ROUND(AVG(minutes_to_acknowledge), 1) AS mtta_minutes,
       ROUND(AVG(minutes_to_resolve), 1)     AS mttr_minutes,
       COUNT_IF(sla_breached)                AS sla_breaches
FROM gold.incident_metrics
GROUP BY ALL
ORDER BY assigned_team, severity;

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ### Scaling note (interview talking point)
-- MAGIC A full rebuild is fine for thousands of rows per hour. At millions you would:
-- MAGIC 1. find the hours touched by the new Silver rows (`SELECT DISTINCT event_hour ...` from the latest batch, or Delta Change Data Feed),
-- MAGIC 2. recompute only those hours, and
-- MAGIC 3. `MERGE` them into `service_health_hourly` on `(service, event_hour)`.
-- MAGIC
-- MAGIC Late events are why step 1 matters: an event for 09:00 arriving at 11:00 must update the 09:00 row.